In [2]:
!pip install category_encoders

  Obtaining dependency information for category_encoders from https://files.pythonhosted.org/packages/a6/06/afcae4dab08612dac244ace7f478543f4fb83bea94177231ef9b4f7bfa06/category_encoders-2.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for scikit-learn>=1.6.0 from https://files.pythonhosted.org/packages/3b/67/be3d369f40d8178ba3bd86635d132e08cb5329b023e4669d9426d84bc007/scikit_learn-1.9.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for joblib>=1.4.0 from https://files.pythonhosted.org/packages/7b/91/984aca2ec129e2757d1e4e3c81c3fcda9d0f85b74670a094cc443d9ee949/joblib-1.5.3-py3-none-any.whl.metadata
  Obtaining dependency information for narwhals>=2.0.1 from https://files.pythonhosted.org/packages/48/ca/36339329c4604adbcc99c899b7eb1ce1a555c499b6a6860757dc9bfed36d/narwhals-2.22.1-py3-none-any.whl.metadata
  Obtaining dependency information for threadpoolctl>=3.5.0 from https://files.pythonhosted.org/packages/32/d5/f9a850d79b0851d1d4ef6456097579a9

In [4]:
!pip install -U scikit-learn category-encoders

In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import category_encoders as ce
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [30]:
df = pd.read_csv("../data/application_train.csv")

In [31]:
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

for col in df.columns:
    print(col)
    print(df[col].value_counts().head())

SK_ID_CURR
SK_ID_CURR
100002    1
337664    1
337661    1
337660    1
337659    1
Name: count, dtype: int64
TARGET
TARGET
0    282686
1     24825
Name: count, dtype: int64
NAME_CONTRACT_TYPE
NAME_CONTRACT_TYPE
Cash loans         278232
Revolving loans     29279
Name: count, dtype: int64
CODE_GENDER
CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64
FLAG_OWN_CAR
FLAG_OWN_CAR
N    202924
Y    104587
Name: count, dtype: int64
FLAG_OWN_REALTY
FLAG_OWN_REALTY
Y    213312
N     94199
Name: count, dtype: int64
CNT_CHILDREN
CNT_CHILDREN
0    215371
1     61119
2     26749
3      3717
4       429
Name: count, dtype: int64
AMT_INCOME_TOTAL
AMT_INCOME_TOTAL
135000.0    35750
112500.0    31019
157500.0    26556
180000.0    24719
90000.0     22483
Name: count, dtype: int64
AMT_CREDIT
AMT_CREDIT
450000.0    9709
675000.0    8877
225000.0    8162
180000.0    7342
270000.0    7241
Name: count, dtype: int64
AMT_ANNUITY
AMT_ANNUITY
9000.0     6385
13500.0    5514
6750.0     

TOTALAREA_MODE
0.0000    582
0.0570    247
0.0547    230
0.0550    227
0.0555    227
Name: count, dtype: int64
WALLSMATERIAL_MODE
WALLSMATERIAL_MODE
Panel           66040
Stone, brick    64815
Block            9253
Wooden           5362
Mixed            2296
Name: count, dtype: int64
EMERGENCYSTATE_MODE
EMERGENCYSTATE_MODE
No     159428
Yes      2328
Name: count, dtype: int64
OBS_30_CNT_SOCIAL_CIRCLE
OBS_30_CNT_SOCIAL_CIRCLE
0.0    163910
1.0     48783
2.0     29808
3.0     20322
4.0     14143
Name: count, dtype: int64
DEF_30_CNT_SOCIAL_CIRCLE
DEF_30_CNT_SOCIAL_CIRCLE
0.0    271324
1.0     28328
2.0      5323
3.0      1192
4.0       253
Name: count, dtype: int64
OBS_60_CNT_SOCIAL_CIRCLE
OBS_60_CNT_SOCIAL_CIRCLE
0.0    164666
1.0     48870
2.0     29766
3.0     20215
4.0     13946
Name: count, dtype: int64
DEF_60_CNT_SOCIAL_CIRCLE
DEF_60_CNT_SOCIAL_CIRCLE
0.0    280721
1.0     21841
2.0      3170
3.0       598
4.0       135
Name: count, dtype: int64
DAYS_LAST_PHONE_CHANGE
DAYS_LAST_PHON

# 1. Missing > 40% features
### EXT_SOURCE_1
Keep with indicators and impute with median. EXT_SOURCE_1 is one of the most predictive credit bureau features in the Home Credit dataset. Dropping this feature would lose valuable information
### OWN_CAR_AGE
Keep with indicators and impute with median. Missing values are probably associated with who don't own a car, and car ownership is a key indicator on the application's economic condition
### Housing-Related  Numeric Features(AVG/MEDI/MODE)
Such as COMMONAREA, NONLIVINGAPARTMENTS, LIVINGAPARTMENTS, LANDAREA, etc.Keep only AVG for representative to reduce redundancy. Keep with indicators and impute with median. These features are also key indicators on applications' economic conditions
### Housing-Related  Categorical Feature
Such as FONDKAPREMONT_MODE, WALLSMATERIAL_MODE, HOUSETYPE_MODE, etc. Keep with indicators and impute with mode. These features may contain important information about the applicants' living conditions


In [32]:
missing_report = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().mean() * 100).round(2),
    'dtype': df.dtypes
}).sort_values('missing_pct', ascending=False)

missing_report = missing_report[missing_report['missing_pct'] > 40]
print('Missing Data Report(>40%):')
print(missing_report)

Missing Data Report(>40%):
                              missing_count  missing_pct    dtype
COMMONAREA_AVG                       214865        69.87  float64
COMMONAREA_MODE                      214865        69.87  float64
COMMONAREA_MEDI                      214865        69.87  float64
NONLIVINGAPARTMENTS_AVG              213514        69.43  float64
NONLIVINGAPARTMENTS_MODE             213514        69.43  float64
NONLIVINGAPARTMENTS_MEDI             213514        69.43  float64
FONDKAPREMONT_MODE                   210295        68.39   object
LIVINGAPARTMENTS_MEDI                210199        68.35  float64
LIVINGAPARTMENTS_AVG                 210199        68.35  float64
LIVINGAPARTMENTS_MODE                210199        68.35  float64
FLOORSMIN_AVG                        208642        67.85  float64
FLOORSMIN_MODE                       208642        67.85  float64
FLOORSMIN_MEDI                       208642        67.85  float64
YEARS_BUILD_AVG                      204488      

# 2. Create missing indicators and AUROC

In [33]:
def create_missing_indicators(df, pct):
    df = df.copy()
    
    missing_cols = df.columns[df.isnull().mean() > pct]
    
    for col in missing_cols:
        df[f"{col}_MISSING"] = df[col].isnull().astype('int')
     
    return df, list(missing_cols)

df, missing_cols = create_missing_indicators(df, 0.05)

missing_indicators = [f"{col}_MISSING" for col in missing_cols] 
X = df[missing_indicators]
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

auroc = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])

print(f'Number of missing indicators: {len(missing_indicators)}')
print(f'AUROC using missing indicators only: {auroc:.4f}')

Number of missing indicators: 58
AUROC using missing indicators only: 0.5868


# 3. Imputation of missing values
### Median for numeric values : Many financial variables are right-skewed and contain outliers. Meidan is more robust than mean and better represents the typical applicants.
### Mode for categorical values: Categorical variables do not have mean or median.

In [34]:
def handle_missing(df):
    df = df.copy()
    
    # Numeric freatures
    num_cols = df.select_dtypes(include='number').columns
    
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())
    
    # Categorical features
    cat_cols = df.select_dtypes(include='object').columns
    
    for col in cat_cols:
        df[col] = df[col].fillna(df[col].mode()[0])
    
    return df

df = handle_missing(df)

In [36]:
df.to_csv("../data/preprocessed_application_train.csv", index=False)